# 05 — Chuẩn bị hai nhánh dữ liệu

Tương ứng src/prepare.py. Ghép 32 biến giá với 12 biến bổ trợ, chọn cùng tập ngày có đủ dữ liệu để so sánh công bằng.

**Cách chạy:** chọn kernel Python rồi chạy từng ô từ trên xuống (Shift+Enter), hoặc chọn **Run All**.

**Cần có trước:** Chạy notebook 02 trước để có fpt_multisource_features.csv; repo đã kèm bản đóng gói để xem thử.

Các ô gọi hàm thật trong `src/`, không gọi `main()` nhận tham số của kernel và không chép lại thuật toán. Mỗi lần chạy có thể cập nhật tệp kết quả trong `data/processed/`, `reports/`, `figures/`, `models/`. Dữ liệu nghiên cứu dừng tại 30/09/2025.

Xem [hướng dẫn](README.md) nếu chạy trên Google Colab hoặc gặp lỗi kernel.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## 4. Tạo và lưu hai bảng đặc trưng

Bảng giá giữ toàn lịch sử dùng được; bảng common chỉ giữ ngày có đủ 44 đầu vào. Không xóa phiên chỉ vì chưa biết nhãn tương lai: phiên cuối vẫn có ích khi dự báo.

In [ ]:
from prepare import prepare, build_experiment_table, BRANCH_PRICE, BRANCH_MACRO
if not SUPPLEMENTAL_PATH.is_file():
    raise FileNotFoundError("Chạy notebook 02 hoặc bước dữ liệu bổ trợ trong 00_RUN_ALL trước.")
feature_rows, labeled_rows = prepare(PRICE_PATH, PROCESSED_DIR / "fpt_next_session_features.csv", SUPPLEMENTAL_PATH)
base, common, branches = build_experiment_table(PRICE_PATH, SUPPLEMENTAL_PATH)
display(pd.DataFrame([{"Nhánh": name, "Số biến": len(columns)} for name, columns in branches.items()]))
display(common[["date", "target_date", "current_close", "next_close", "direction"]].tail())

## 5. Xem từng nhánh dùng cột nào

Giữ cùng hàng cho hai nhánh. Phần khác biệt là 12 cột bổ trợ, không phải chọn một giai đoạn Test dễ hơn cho một nhánh.

In [ ]:
for name, columns in branches.items():
    display(Markdown(f"### {name} — {len(columns)} biến"))
    display(pd.DataFrame({"Đặc trưng đầu vào": columns}))
print("Bảng giá:", len(base), "| Cửa sổ chung (gồm phiên chưa có nhãn):", len(common))

## 6. Kiểm tra nhãn và dữ liệu thiếu

Mỗi mẫu hồi quy cần next_close; phân loại còn cần direction (phiên đứng giá có direction thiếu theo thiết kế). Thiếu đầu vào và chưa biết nhãn là hai việc khác nhau.

In [ ]:
assert not common[branches[BRANCH_MACRO]].isna().any().any()
print("Mẫu có nhãn giá:", int(common["next_close"].notna().sum()))
print("Mẫu có nhãn tăng/giảm:", int(common["direction"].notna().sum()))
display(common[["date", "target_date", "current_close", "next_close", "direction"]].tail(10))

## 7. Xem trước mốc chia thời gian

Chia 85% đầu / 15% cuối theo ngày nhãn; không fit mô hình ở notebook này. Đầu vào tương lai không được dùng để học tham số chuẩn hóa.

In [ ]:
from train import _chronological_splits
masks, boundary = _chronological_splits(common)
display(pd.Series(boundary).to_frame("Train/Test"))
partition_rows = []
for part, mask in masks.items():
    subset = common.loc[mask]
    partition_rows.append({
        "Tập": part, "Mẫu hồi quy": len(subset),
        "Mẫu phân loại": int(subset["direction"].notna().sum()),
        "Ngày nhãn đầu": subset["target_date"].min(),
        "Ngày nhãn cuối": subset["target_date"].max(),
    })
display(pd.DataFrame(partition_rows))
assert common.loc[masks["train"], "target_date"].max() < common.loc[masks["test"], "target_date"].min()

**Đầu ra:** data/processed/fpt_next_session_features.csv và fpt_two_branch_features.csv. Tiếp theo: 06_train.ipynb.